In [2]:
from typing import Annotated
from langgraph.graph import StateGraph,START
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image,display
import gradio as gr
from langgraph.prebuilt import ToolNode,tools_condition
import requests
import os
from langchain_openai import ChatOpenAI
from typing import TypedDict

In [3]:
load_dotenv(override=True)

True

In [4]:
from langchain_community.utilities import GoogleSerperAPIWrapper
serper = GoogleSerperAPIWrapper()
serper.run("what are thebsteps for agentic evalution")

ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
from langchain.agents import Tool
tool_search = Tool(
                name = "search",
                func = serper.run,
                description = "useful for online search"
)

In [ ]:
tool_search.invoke("agent monitoring methods")

In [ ]:
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"
def push(text:str):
    requests.post(pushover_url,data = {"token": pushover_token, "user": pushover_user, "message": text})

In [ ]:
tool_push = Tool(
        name = "send push notification",
        func = push,
        description = "send a notification"
)

In [10]:
tools = [tool_search,tool_push]

NameError: name 'tool_search' is not defined

In [6]:
#1 define state object
class State(TypedDict):
    messages:Annotated[list,add_messages]

In [7]:
#2 start the graph builder with state class
graph_builder=StateGraph(State)

In [9]:
llm=ChatOpenAI(model="gpt-4o-mini")
llm_with_tools=llm.bind_tools(tools)

NameError: name 'tools' is not defined

In [ ]:
#3 create node
def chatbot(state:State):
    return {"messages" : [llm_with_tools.invoke(state["messages"])}

grapg_builder.add_node("chatbot",chatbot)
graph_builder.add_node("tools,ToolNode(tools=tools))

In [ ]:
#create edges
graph_builder.add_conditional_edges("chatbot",tools_condition,"tools")
graph_builder.add_edge("tools","chatbot")
graph_builder.add_edge(START,"chatbot")

In [ ]:
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
def chat(user_input:str,history):
    result = graph.invoke({"messages" :[{"role":"user","content",user_input}]})
    return result["messages"][-1].content
gr.ChatInterface(chat).launch(share=True)